# 🏛️ ADOIT Data Product Pipeline
## Domain: Enterprise Architecture | Catalog: `adoit_product` | Owner: EA Team

This notebook demonstrates the **ADOIT Data Product** — owned and managed entirely by the Enterprise Architecture team within their own catalog (`adoit_product`).

**Data Product Characteristics shown here:**
- ✅ Clear Owner — EA Team owns the catalog
- ✅ Business Purpose — Application landscape for rationalization and investment planning
- ✅ Documented Definitions — Every column has a COMMENT
- ✅ Quality Checks — Completeness scoring, validation rules
- ✅ Lineage — Bronze → Silver → Gold transformation chain
- ✅ Reliability — SLA defined in TBLPROPERTIES

**Data Objects:**
- `raw_applications` — 10 enterprise applications from ADOIT
- `raw_business_capabilities` — 10 business capabilities
- `raw_app_capability_map` — 13 app-to-capability mappings

## 🪨 Bronze Layer — Raw Ingestion
Raw data as extracted from ADOIT REST API. No transformations, just landed with an `ingested_at` timestamp.

In [0]:
# ── Bootstrap: Create bronze tables and load CSV data ──
# This cell ensures tables exist when running via CI/CD (clean rebuild)
import os

# Determine the data directory (relative to this notebook in the bundle)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
bundle_root = notebook_path.rsplit('/src/notebooks/', 1)[0]
data_dir = f"/Workspace{bundle_root}/src/data"

# If running interactively (not from bundle), try the repo path
if not os.path.exists(data_dir):
    data_dir = "/Workspace/Users/skarotirajashekar@godevsuite060.onmicrosoft.com/data-mesh-demo/src/data"

print(f"Data directory: {data_dir}")

# Load ADOIT CSVs into bronze tables
for csv_file, table_name in [
    ("adoit_applications.csv", "adoit_product.bronze.raw_applications"),
    ("adoit_capabilities.csv", "adoit_product.bronze.raw_business_capabilities"),
    ("adoit_app_capability_map.csv", "adoit_product.bronze.raw_app_capability_map"),
]:
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/{csv_file}")
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f"  ✓ {table_name} ({df.count()} rows)")

print("Bronze tables loaded.")

In [0]:
%sql
-- Create silver and gold tables (empty) so MERGE INTO has targets
-- Silver: applications
CREATE TABLE IF NOT EXISTS adoit_product.silver.applications (
  app_id STRING, app_name STRING, app_owner STRING, department STRING,
  lifecycle_status STRING, criticality STRING, technology_stack STRING,
  go_live_date DATE, is_cloud_hosted BOOLEAN, app_age_years DOUBLE,
  quality_score DOUBLE, ingested_at TIMESTAMP
);

-- Silver: business_capabilities (from raw capabilities)
CREATE TABLE IF NOT EXISTS adoit_product.silver.business_capabilities (
  capability_id STRING, capability_name STRING, capability_domain STRING,
  business_value STRING, maturity_level STRING, ingested_at TIMESTAMP
);

-- Gold: application_landscape
CREATE TABLE IF NOT EXISTS adoit_product.gold.application_landscape (
  app_id STRING, app_name STRING, app_owner STRING, department STRING,
  lifecycle_status STRING, criticality STRING, technology_stack STRING,
  go_live_date DATE, is_cloud_hosted BOOLEAN, app_age_years DOUBLE,
  capability_count INT, primary_capabilities STRING, all_capabilities STRING,
  quality_score DOUBLE, product_updated_at TIMESTAMP
);

In [0]:
%sql
-- Bronze: Raw application register from ADOIT
SELECT * FROM adoit_product.bronze.raw_applications ORDER BY app_id;

In [0]:
%sql
-- Bronze: Business capability taxonomy from ADOIT
SELECT * FROM adoit_product.bronze.raw_business_capabilities ORDER BY capability_id;

## ⚙️ Silver Layer — Curated & Quality-Scored
Transformations applied:
1. **Standardization** — `TRIM()`, `INITCAP()` on text fields
2. **Enrichment** — `app_age_years` calculated, `is_cloud_hosted` derived from technology stack
3. **Quality Scoring** — Each record scored 0–1.0 based on field completeness
4. **Validation** — Only records with valid `app_id` pass through

In [0]:
%sql
-- Silver transformation: cleanse, enrich, and quality-score
-- This is the actual transformation that would run on schedule
MERGE INTO adoit_product.silver.applications AS target
USING (
    SELECT 
        app_id,
        TRIM(app_name) AS app_name,
        app_owner,
        INITCAP(department) AS department,
        lifecycle_status,
        criticality,
        technology_stack,
        go_live_date,
        ROUND(DATEDIFF(CURRENT_DATE(), go_live_date) / 365.25, 1) AS app_age_years,
        CASE 
            WHEN LOWER(technology_stack) LIKE '%cloud%' 
              OR LOWER(technology_stack) LIKE '%365%'
              OR LOWER(technology_stack) LIKE '%databricks%'
            THEN TRUE ELSE FALSE 
        END AS is_cloud_hosted,
        -- Quality score: completeness of key fields (0 to 1.0)
        ROUND((
            CASE WHEN app_name IS NOT NULL THEN 0.2 ELSE 0 END +
            CASE WHEN app_owner IS NOT NULL THEN 0.2 ELSE 0 END +
            CASE WHEN department IS NOT NULL THEN 0.2 ELSE 0 END +
            CASE WHEN go_live_date IS NOT NULL THEN 0.2 ELSE 0 END +
            CASE WHEN technology_stack IS NOT NULL THEN 0.2 ELSE 0 END
        ), 2) AS quality_score,
        current_timestamp() AS processed_at
    FROM adoit_product.bronze.raw_applications
    WHERE app_id IS NOT NULL AND app_name IS NOT NULL
) AS source
ON target.app_id = source.app_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Silver transformation for business capabilities
MERGE INTO adoit_product.silver.business_capabilities AS target
USING (
    SELECT 
        capability_id, capability_name, capability_domain,
        business_value, maturity_level,
        current_timestamp() AS ingested_at
    FROM adoit_product.bronze.raw_business_capabilities
) AS source
ON target.capability_id = source.capability_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Silver: Curated applications with quality scores and derived fields
SELECT app_id, app_name, app_owner, department, lifecycle_status, criticality,
       is_cloud_hosted, app_age_years, quality_score
FROM adoit_product.silver.applications
ORDER BY quality_score DESC;

## 🏆 Gold Layer — Business-Ready Data Product
The **Application Landscape** data product joins applications with their business capabilities to create a 360-degree view. This is what consumers (CTO Office, Risk Team) actually query.

In [0]:
%sql
-- Gold: Build the Application Landscape data product
-- Joins silver applications with capability mappings
MERGE INTO adoit_product.gold.application_landscape AS target
USING (
    SELECT 
        a.app_id, a.app_name, a.app_owner, a.department,
        a.lifecycle_status, a.criticality, a.technology_stack,
        a.is_cloud_hosted, a.app_age_years,
        COUNT(DISTINCT m.capability_id) AS capability_count,
        CONCAT_WS(', ', COLLECT_SET(CASE WHEN m.support_level = 'Primary' THEN c.capability_name END)) AS primary_capabilities,
        CONCAT_WS(', ', COLLECT_SET(c.capability_name)) AS all_capabilities,
        a.quality_score,
        current_timestamp() AS product_generated_at
    FROM adoit_product.silver.applications a
    LEFT JOIN adoit_product.bronze.raw_app_capability_map m ON a.app_id = m.app_id
    LEFT JOIN adoit_product.silver.business_capabilities c ON m.capability_id = c.capability_id
    GROUP BY a.app_id, a.app_name, a.app_owner, a.department, a.lifecycle_status, 
             a.criticality, a.technology_stack, a.is_cloud_hosted, a.app_age_years, a.quality_score
) AS source
ON target.app_id = source.app_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- The final data product: Application Landscape
SELECT app_id, app_name, app_owner, department, criticality, lifecycle_status,
       is_cloud_hosted, ROUND(app_age_years, 1) AS age_yrs, capability_count,
       primary_capabilities, quality_score
FROM adoit_product.gold.application_landscape
ORDER BY criticality, app_name;

## 📊 Quality Checks & Data Product Metadata
Every good data product has embedded quality rules and rich metadata.

In [0]:
%sql
-- Data quality validation rules for ADOIT product
SELECT 
    'Completeness: app_owner' AS check_name,
    ROUND(COUNT(CASE WHEN app_owner IS NOT NULL THEN 1 END) * 100.0 / COUNT(*), 1) AS score_pct,
    CASE WHEN COUNT(CASE WHEN app_owner IS NOT NULL THEN 1 END) * 100.0 / COUNT(*) >= 95 THEN 'PASS' ELSE 'FAIL' END AS status
FROM adoit_product.gold.application_landscape
UNION ALL
SELECT 'Uniqueness: app_id',
    CASE WHEN COUNT(*) = COUNT(DISTINCT app_id) THEN 100.0 ELSE ROUND(COUNT(DISTINCT app_id) * 100.0 / COUNT(*), 1) END,
    CASE WHEN COUNT(*) = COUNT(DISTINCT app_id) THEN 'PASS' ELSE 'FAIL' END
FROM adoit_product.gold.application_landscape
UNION ALL
SELECT 'Validity: criticality values',
    ROUND(COUNT(CASE WHEN criticality IN ('Critical', 'High', 'Medium', 'Low') THEN 1 END) * 100.0 / COUNT(*), 1),
    CASE WHEN COUNT(CASE WHEN criticality NOT IN ('Critical', 'High', 'Medium', 'Low') THEN 1 END) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM adoit_product.gold.application_landscape
UNION ALL
SELECT 'Avg Quality Score',
    ROUND(AVG(quality_score) * 100, 1),
    CASE WHEN AVG(quality_score) >= 0.8 THEN 'PASS' ELSE 'FAIL' END
FROM adoit_product.gold.application_landscape;

In [0]:
%sql
-- Data product metadata embedded in Unity Catalog
SHOW TBLPROPERTIES adoit_product.gold.application_landscape;